In [5]:
# %load_ext and %env for notebook HMR (can be skipped if not using Jupyter)
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

# Imports
import pandas as pd
import numpy as np
import zarr
import tifffile as tiff
import geopandas as gpd
from shapely.geometry import Polygon
from shapely import vectorized  # Requires Shapely ≥ 2.0
import celldega as dega


# ----------------------------
# Utility Functions
# ----------------------------

def open_zarr(path: str) -> zarr.Group:
    store = zarr.ZipStore(path, mode="r") if path.endswith(".zip") else zarr.DirectoryStore(path)
    return zarr.group(store=store)

def get_polygon_from_image_region(image, transformation_matrix, region_size=1000):
    x_min, y_min = image.shape[2] / 2, image.shape[1] / 2
    x_max, y_max = x_min + region_size, y_min + region_size
    corners_img = np.array([
        [x_min, y_min, 1],
        [x_max, y_min, 1],
        [x_max, y_max, 1],
        [x_min, y_max, 1]
    ])
    T_inv = np.linalg.inv(transformation_matrix)
    corners_micron = (T_inv @ corners_img.T).T[:, :2]
    return Polygon(corners_micron), (x_min, y_min)

def filter_df_within_polygon(df, x_col, y_col, polygon):
    mask = vectorized.contains(polygon, df[x_col].values, df[y_col].values)
    return df[mask].copy()


# ----------------------------
# Load transformation matrix
# ----------------------------

try:
    root = open_zarr("data/Xenium_V1_human_Pancreas_FFPE_outs/cells.zarr")
    transformation_matrix = root["masks"]["homogeneous_transform"][:3, :3]
except KeyError as e:
    raise KeyError(f"Could not find the transformation matrix in the Zarr file: {e}") from e


# ----------------------------
# Load one image to define region
# ----------------------------

image = tiff.imread("data/Xenium_V1_human_Pancreas_FFPE_outs/morphology_focus/morphology_focus_0000.ome.tif")
region_polygon, (x_min, y_min) = get_polygon_from_image_region(image, transformation_matrix)


# ----------------------------
# Filter and save transcripts
# ----------------------------

df_transcripts = pd.read_parquet("data/Xenium_V1_human_Pancreas_FFPE_outs/transcripts.parquet")
df_transcripts_subset = filter_df_within_polygon(df_transcripts, "x_location", "y_location", region_polygon)

# Reset coordinates to (0,0)
df_transcripts_subset["x_location"] -= region_polygon.bounds[0]
df_transcripts_subset["y_location"] -= region_polygon.bounds[1]

df_transcripts_subset.to_parquet("data/unit_test_sample_data/Xenium_V1_human_Pancreas_FFPE_outs/transcripts.parquet")


# ----------------------------
# Filter and save cell boundaries
# ----------------------------

df_boundaries = pd.read_parquet("data/Xenium_V1_human_Pancreas_FFPE_outs/cell_boundaries.parquet")
df_boundaries_subset = filter_df_within_polygon(df_boundaries, "vertex_x", "vertex_y", region_polygon)

df_boundaries_subset["vertex_x"] -= region_polygon.bounds[0]
df_boundaries_subset["vertex_y"] -= region_polygon.bounds[1]

df_boundaries_subset.to_parquet("data/unit_test_sample_data/Xenium_V1_human_Pancreas_FFPE_outs/cell_boundaries.parquet")


# ----------------------------
# Crop and save morphology images
# ----------------------------

for i in range(4):
    image = tiff.imread(f"data/Xenium_V1_human_Pancreas_FFPE_outs/morphology_focus/morphology_focus_000{i}.ome.tif")
    x0, y0 = int(x_min), int(y_min)
    x1, y1 = x0 + 1000, y0 + 1000
    cropped = image[:, y0:y1, x0:x1]
    tiff.imwrite(f"data/unit_test_sample_data/Xenium_V1_human_Pancreas_FFPE_outs/morphology_focus/morphology_focus_000{i}.ome.tif", cropped)


# ----------------------------
# Filter and save cells
# ----------------------------

cells_df = pd.read_parquet("data/Xenium_V1_human_Pancreas_FFPE_outs/cells.parquet")
df_cells_subset = filter_df_within_polygon(cells_df, "x_centroid", "y_centroid", region_polygon)

df_cells_subset["x_centroid"] -= region_polygon.bounds[0]
df_cells_subset["y_centroid"] -= region_polygon.bounds[1]

df_cells_subset.to_parquet("data/unit_test_sample_data/Xenium_V1_human_Pancreas_FFPE_outs/cells.parquet")
df_cells_subset.to_csv("data/unit_test_sample_data/Xenium_V1_human_Pancreas_FFPE_outs/cells.csv", index=False)
df_cells_subset.to_csv("data/unit_test_sample_data/Xenium_V1_human_Pancreas_FFPE_outs/cells.csv.gz", index=False, compression="gzip")


# ----------------------------
# Filter and save clusters based on retained cell_ids
# ----------------------------

clusters = pd.read_csv("data/unit_test_sample_data/Xenium_V1_human_Pancreas_FFPE_outs/analysis/clustering/gene_expression_graphclust/clusters.csv")
filtered_clusters = clusters[clusters["Barcode"].isin(df_cells_subset["cell_id"])]
filtered_clusters.to_csv("data/unit_test_sample_data/Xenium_V1_human_Pancreas_FFPE_outs/analysis/clustering/gene_expression_graphclust/clusters.csv", index=False)

<tifffile.TiffFile 'morphology_focus_0000.ome.tif'> OME series cannot read multi-file pyramids


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
env: ANYWIDGET_HMR=1


<tifffile.TiffFile 'morphology_focus_0000.ome.tif'> OME series cannot read multi-file pyramids
<tifffile.TiffFile 'morphology_focus_0001.ome.tif'> OME series cannot read multi-file pyramids
<tifffile.TiffFile 'morphology_focus_0002.ome.tif'> OME series cannot read multi-file pyramids
<tifffile.TiffFile 'morphology_focus_0003.ome.tif'> OME series cannot read multi-file pyramids


In [7]:
sample = 'Xenium_V1_human_Pancreas_FFPE_outs'
data_dir = f'data/unit_test_sample_data/{sample}'
path_landscape_files=f'data/UNIT_TEST_landscape_files/{sample}'

tile_size=250
image_tile_layer='dapi'


dega.pre.main(
    sample=sample,
    data_root_dir=data_dir,
    tile_size=tile_size,
    image_tile_layer=image_tile_layer,
    path_landscape_files=path_landscape_files,
    use_int_index=True,
    )

Starting preprocessing for sample: Xenium_V1_human_Pancreas_FFPE_outs

========Unzip and extract Xenium-related files========
All files have been successfully extracted or skipped.

========Write xenium transform file from the Zarr folder========
Transformation matrix saved to 'data/UNIT_TEST_landscape_files/Xenium_V1_human_Pancreas_FFPE_outs/micron_to_image_transform.csv'.

========Check if all required files or directories exist========
All required files or directories for technology 'Xenium' are present in 'data/unit_test_sample_data/Xenium_V1_human_Pancreas_FFPE_outs'.

========Make meta cells in pixel space========
Done.

========Create cluster gene expression (df_sig)========
Cluster-specific gene expression signatures saved successfully.

========Write meta gene files========
cbg is a dense DataFrame. Proceeding with dense operations.
Calculating mean expression
Calculating variance
All meta gene files are succesfully saved.
data/UNIT_TEST_landscape_files/Xenium_V1_human_Pancre

/Users/jishar/anaconda3/envs/celldega_env_latest/lib/python3.11/site-packages/skimage/_shared/utils.py:328: UserWarning: /Users/jishar/Documents/celldega/notebooks/data/UNIT_TEST_landscape_files/Xenium_V1_human_Pancreas_FFPE_outs/dapi_output_regular.tif is a low contrast image
  return func(*args, **kwargs)


Image tiles created successfully.

========Generating transcript tiles========


Processing chunks: 100%|███████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 291.43it/s]
Processing coarse tiles: 1tile [00:00, 20.41tile/s]


tile bounds: {'x_min': 0, 'x_max': 999.95, 'y_min': 0, 'y_max': 999.99}

========Generating boundary tiles========

========Create cell boundary spatial tiles========
technology Xenium


Processing coarse tiles: 100%|██████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 22.27it/s]

Done.

========Save landscape parameters========
Done.
Preprocessing completed successfully.


In [8]:
landscape_ist = dega.viz.Landscape(
    technology='Xenium',
    base_url = f"http://localhost:{dega.viz.get_local_server()}/{path_landscape_files}",
)

landscape_ist

Server running on port 56155


Landscape(base_url='http://localhost:56155/data/UNIT_TEST_landscape_files/Xenium_V1_human_Pancreas_FFPE_outs',…